# Task 8 — Multi-Agent AI System for Report Generation

Multiple specialized agents collaborate step by step to turn a single user
question into a structured report, instead of relying on one LLM call.

## Objectives checklist
- [x] Planner Agent
- [x] Research Agent
- [x] Writer Agent
- [x] Summary Agent
- [x] Evaluator Agent (optional, on by default, can be disabled)
- [x] Agents work together step by step, each consuming the previous agent's output
- [x] Automated tests proving the hand-off order and error handling

## About `USE_LIVE_MODEL`
Set the flag below to `True` to download and use a real Hugging Face model
(`google/flan-t5-base`) -- requires internet access. Left `False`, the
notebook uses a small templated offline stand-in per agent role so the full
5-agent pipeline can be demonstrated and verified without downloading
anything, which is what was used to produce the outputs saved in this
notebook.

In [1]:
%pip install -q transformers torch

Note: you may need to restart the kernel to use updated packages.


## 1. Configuration

In [2]:
USE_LIVE_MODEL = False  # Set True to use a real Hugging Face LLM (requires internet)


## 2. Shared LLM backend

All agents share one backend and differ only in the prompt/role they use.

In [3]:
import re


def offline_agent_llm(prompt: str) -> str:
    """Small templated offline stand-in that behaves differently per agent role,
    so the multi-agent hand-off can be demonstrated without downloading a model."""
    lower = prompt.lower()
    if "planning agent" in lower:
        topic = prompt.split("Question:", 1)[1].split("\n\nPlan:")[0].strip()
        return (
            f"1. Define and scope the topic: {topic}\n"
            "2. Identify the key benefits and challenges\n"
            "3. Gather supporting examples or data\n"
            "4. Draw conclusions and recommendations"
        )
    if "research agent" in lower:
        plan = prompt.split("Plan:", 1)[1].split("\n\nResearch notes:")[0].strip()
        return "Research notes covering each planned point:\n" + "\n".join(
            f"- {line.strip()}: relevant supporting information gathered." for line in plan.split("\n") if line.strip()
        )
    if "writer agent" in lower:
        question = prompt.split("Question:", 1)[1].split("\n\nResearch notes:")[0].strip()
        return (
            f"Introduction: This report addresses the question \"{question}\".\n\n"
            "Body: Based on the research notes, several key points emerge that directly answer the question, "
            "supported by the gathered evidence.\n\n"
            "Conclusion: In summary, the findings provide a clear, well-supported answer to the original question."
        )
    if "summary agent" in lower:
        return "Executive summary: the report answers the question with a clear introduction, evidence-based body, and actionable conclusion."
    if "evaluator agent" in lower:
        return "Evaluation: the report is well-structured and on-topic; consider adding more quantitative data for stronger support."
    return "Okay."


def build_live_generator(model_name: str = "google/flan-t5-base", max_new_tokens: int = 256):
    from transformers import pipeline
    pipe = pipeline("text2text-generation", model=model_name, max_new_tokens=max_new_tokens)
    return lambda prompt: pipe(prompt)[0]["generated_text"].strip()


class LLMBackend:
    def __init__(self, generator=None):
        self.generator = generator or (build_live_generator() if USE_LIVE_MODEL else offline_agent_llm)

    def run(self, prompt: str) -> str:
        return self.generator(prompt)


print(f"LLMBackend ready (USE_LIVE_MODEL={USE_LIVE_MODEL}).")

LLMBackend ready (USE_LIVE_MODEL=False).


## 3. The five agents

In [4]:
class PlannerAgent:
    def __init__(self, llm: LLMBackend):
        self.llm = llm

    def run(self, question: str) -> str:
        prompt = (
            "You are a planning agent. Break the following question into a "
            "numbered list of 3-4 key research points to investigate.\n\n"
            f"Question: {question}\n\nPlan:"
        )
        return self.llm.run(prompt)


class ResearchAgent:
    def __init__(self, llm: LLMBackend):
        self.llm = llm

    def run(self, question: str, plan: str) -> str:
        prompt = (
            "You are a research agent. Given the question and the research "
            "plan below, provide concise factual notes covering each point.\n\n"
            f"Question: {question}\n\nPlan:\n{plan}\n\nResearch notes:"
        )
        return self.llm.run(prompt)


class WriterAgent:
    def __init__(self, llm: LLMBackend):
        self.llm = llm

    def run(self, question: str, research_notes: str) -> str:
        prompt = (
            "You are a writer agent. Using the research notes below, write a "
            "well-structured report with an introduction, body, and conclusion "
            "that answers the original question.\n\n"
            f"Question: {question}\n\nResearch notes:\n{research_notes}\n\nReport:"
        )
        return self.llm.run(prompt)


class SummaryAgent:
    def __init__(self, llm: LLMBackend):
        self.llm = llm

    def run(self, report: str) -> str:
        prompt = (
            "You are a summary agent. Summarize the following report in 2-3 "
            "sentences for an executive audience.\n\n"
            f"Report:\n{report}\n\nSummary:"
        )
        return self.llm.run(prompt)


class EvaluatorAgent:
    def __init__(self, llm: LLMBackend):
        self.llm = llm

    def run(self, question: str, report: str) -> str:
        prompt = (
            "You are an evaluator agent. Review the report below for how well "
            "it answers the original question. List any gaps or suggested "
            "improvements in 2-3 bullet points.\n\n"
            f"Question: {question}\n\nReport:\n{report}\n\nEvaluation:"
        )
        return self.llm.run(prompt)


print("All 5 agents defined.")

All 5 agents defined.


## 4. Orchestrator

Runs the agents in sequence, each consuming the previous agent's output.

In [5]:
class ReportOrchestrator:
    def __init__(self, use_evaluator: bool = True, llm: LLMBackend = None):
        self.llm = llm or LLMBackend()
        self.planner = PlannerAgent(self.llm)
        self.researcher = ResearchAgent(self.llm)
        self.writer = WriterAgent(self.llm)
        self.summarizer = SummaryAgent(self.llm)
        self.evaluator = EvaluatorAgent(self.llm) if use_evaluator else None

    def generate_report(self, question: str) -> dict:
        if not question or not question.strip():
            raise ValueError("Question cannot be empty.")

        plan = self.planner.run(question)
        research_notes = self.researcher.run(question, plan)
        report = self.writer.run(question, research_notes)
        summary = self.summarizer.run(report)

        result = {
            "question": question,
            "plan": plan,
            "research_notes": research_notes,
            "report": report,
            "summary": summary,
        }
        if self.evaluator:
            result["evaluation"] = self.evaluator.run(question, report)
        return result


print("ReportOrchestrator ready.")

ReportOrchestrator ready.


## 5. Demo run

In [6]:
orchestrator = ReportOrchestrator(use_evaluator=True)
result = orchestrator.generate_report("What are the benefits of remote work for small businesses?")

print("=== PLAN (Planner Agent) ===")
print(result["plan"])
print("\n=== RESEARCH NOTES (Research Agent) ===")
print(result["research_notes"])
print("\n=== REPORT (Writer Agent) ===")
print(result["report"])
print("\n=== SUMMARY (Summary Agent) ===")
print(result["summary"])
print("\n=== EVALUATION (Evaluator Agent) ===")
print(result["evaluation"])

=== PLAN (Planner Agent) ===
1. Define and scope the topic: What are the benefits of remote work for small businesses?
2. Identify the key benefits and challenges
3. Gather supporting examples or data
4. Draw conclusions and recommendations

=== RESEARCH NOTES (Research Agent) ===
Research notes covering each planned point:
- 1. Define and scope the topic: What are the benefits of remote work for small businesses?: relevant supporting information gathered.
- 2. Identify the key benefits and challenges: relevant supporting information gathered.
- 3. Gather supporting examples or data: relevant supporting information gathered.
- 4. Draw conclusions and recommendations: relevant supporting information gathered.

=== REPORT (Writer Agent) ===
Introduction: This report addresses the question "What are the benefits of remote work for small businesses?".

Body: Based on the research notes, several key points emerge that directly answer the question, supported by the gathered evidence.

Conclu

## 6. Automated tests (offline)

A fake LLM returns a distinct, traceable output per agent role, so these tests prove each agent's prompt genuinely contains the previous agent's output (the hand-off chain is real), plus the optional-evaluator and empty-question behavior.

In [7]:
def fake_llm(prompt: str) -> str:
    if "planning agent" in prompt.lower():
        return "PLAN: [step1, step2, step3]"
    if "research agent" in prompt.lower():
        assert "PLAN:" in prompt, "Research agent should receive the planner\'s output"
        return "RESEARCH_NOTES: based on PLAN"
    if "writer agent" in prompt.lower():
        assert "RESEARCH_NOTES:" in prompt, "Writer agent should receive the research notes"
        return "REPORT: built from RESEARCH_NOTES"
    if "summary agent" in prompt.lower():
        assert "REPORT:" in prompt, "Summary agent should receive the report"
        return "SUMMARY: of REPORT"
    if "evaluator agent" in prompt.lower():
        assert "REPORT:" in prompt, "Evaluator agent should receive the report"
        return "EVALUATION: of REPORT"
    raise AssertionError(f"Unexpected prompt with no recognizable agent role: {prompt[:80]}")


def test_agents_run_in_order_with_evaluator():
    o = ReportOrchestrator(use_evaluator=True, llm=LLMBackend(generator=fake_llm))
    r = o.generate_report("What are the benefits of remote work?")
    assert r["plan"] == "PLAN: [step1, step2, step3]"
    assert r["research_notes"] == "RESEARCH_NOTES: based on PLAN"
    assert r["report"] == "REPORT: built from RESEARCH_NOTES"
    assert r["summary"] == "SUMMARY: of REPORT"
    assert r["evaluation"] == "EVALUATION: of REPORT"
    print("PASS test_agents_run_in_order_with_evaluator")


def test_evaluator_is_optional():
    o = ReportOrchestrator(use_evaluator=False, llm=LLMBackend(generator=fake_llm))
    r = o.generate_report("What are the benefits of remote work?")
    assert "evaluation" not in r
    assert o.evaluator is None
    print("PASS test_evaluator_is_optional")


def test_empty_question_is_rejected():
    o = ReportOrchestrator(llm=LLMBackend(generator=fake_llm))
    try:
        o.generate_report("   ")
    except ValueError:
        print("PASS test_empty_question_is_rejected")
        return
    raise AssertionError("Expected ValueError for empty question")


def test_all_four_required_agents_exist():
    o = ReportOrchestrator(llm=LLMBackend(generator=fake_llm))
    assert o.planner is not None
    assert o.researcher is not None
    assert o.writer is not None
    assert o.summarizer is not None
    print("PASS test_all_four_required_agents_exist")


test_agents_run_in_order_with_evaluator()
test_evaluator_is_optional()
test_empty_question_is_rejected()
test_all_four_required_agents_exist()
print("\nAll multi-agent orchestrator tests passed.")

PASS test_agents_run_in_order_with_evaluator
PASS test_evaluator_is_optional
PASS test_empty_question_is_rejected
PASS test_all_four_required_agents_exist

All multi-agent orchestrator tests passed.


## 7. Interactive mode (optional)

Run this cell in a real Jupyter session to build a report on your own question. Exits gracefully if there is no input available (e.g. run non-interactively).

In [8]:
try:
    while True:
        question = input("Enter a question to build a report on (or \'exit\'): ").strip()
        if not question or question.lower() in ("exit", "quit"):
            break
        r = orchestrator.generate_report(question)
        print("\n=== REPORT ===")
        print(r["report"])
        print("\n=== SUMMARY ===")
        print(r["summary"])
        print()
except Exception:
    print("(No interactive input available -- skipping interactive mode.)")

(No interactive input available -- skipping interactive mode.)


## Notes
- All agents share one `LLMBackend` to keep the system self-contained; each agent uses a different prompt/role rather than a different model. `LLMBackend(generator=...)` accepts any `callable(prompt) -> str`, which is how the tests substitute a fake model.
- To use a stronger model or an API-based LLM (OpenAI/Anthropic), only `LLMBackend` needs to change -- the agent roles and orchestration stay the same.
